# Fine-tune Llama 3.2 1B for Vietnamese Medical QA

This notebook fine-tunes `unsloth/Llama-3.2-1B-Instruct-bnb-4bit` on `hungnm/vietnamese-medical-qa` with Unsloth LoRA adapters, `TrainingArguments`, and early stopping so the resulting adapter can be served by this repo's `medqa-lora` deployment flow.

In [ ]:
%pip install torch==2.10.0 --index-url https://download.pytorch.org/whl/cu130
%pip install unsloth trl datasets huggingface_hub peft

In [ ]:
import torch
torch.__version__

In [ ]:
import os
from pathlib import Path
import torch
import unsloth
from datasets import DatasetDict, load_dataset
from huggingface_hub import login
from transformers import DataCollatorForLanguageModeling, EarlyStoppingCallback, TrainingArguments, set_seed
from trl.trainer.sft_trainer import SFTTrainer
from unsloth import FastLanguageModel

set_seed(42)

MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"
DATASET_NAME = "hungnm/vietnamese-medical-qa"
WORKDIR = Path.cwd().resolve()
PROJECT_ROOT = WORKDIR.parent if WORKDIR.name == "notebooks" else WORKDIR
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "medqa-lora-llama32-1b"
MAX_SEQ_LENGTH = 2048
TRAIN_TEST_SPLIT = 0.05
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_THRESHOLD = 0.0
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
SYSTEM_PROMPT = (
    "Bạn là trợ lý AI hỗ trợ thông tin y khoa bằng tiếng Việt. "
    "Trả lời rõ ràng, thận trọng, không chẩn đoán quá mức, "
    "và khuyên người dùng gặp bác sĩ khi có dấu hiệu nguy hiểm."
)

HF_TOKEN = os.getenv("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"BF16 available: {USE_BF16}")
print(f"Unsloth version: {getattr(unsloth, '__version__', 'unknown')}")
print(f"Artifacts directory: {OUTPUT_DIR}")

In [ ]:
raw_dataset: DatasetDict = load_dataset(DATASET_NAME)
split_dataset = raw_dataset["train"].train_test_split(test_size=TRAIN_TEST_SPLIT, seed=42)
dataset = DatasetDict({
    "train": split_dataset["train"],
    "val": split_dataset["test"],
})

print(dataset)
sample = dataset["train"][0]
print("\nQuestion preview:\n")
print(sample["question"][:500])
print("\nAnswer preview:\n")
print(sample["answer"][:500])

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=torch.bfloat16,
 )

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "up_proj",
        "down_proj",
        "o_proj",
        "gate_proj",
    ],
    use_rslora=True,
    use_gradient_checkpointing="unsloth",
    random_state=42,
 )
model.config.use_cache = False
model.print_trainable_parameters()

In [ ]:
def build_messages(question, answer=None):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question.strip()},
    ]
    if answer is not None:
        messages.append({"role": "assistant", "content": answer.strip()})
    return messages

def render_conversation(question, answer=None, add_generation_prompt=False):
    messages = build_messages(question, answer)
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )

    prompt = (
        f"{SYSTEM_PROMPT}\n\n"
        f"Câu hỏi:\n{question.strip()}\n\n"
        "Trả lời:\n"
    )
    if answer is None:
        return prompt
    return f"{prompt}{answer.strip()}{tokenizer.eos_token}"

def format_batch(batch):
    texts = [
        render_conversation(question, answer)
        for question, answer in zip(batch["question"], batch["answer"])
    ]
    return {"text": texts}

formatted_dataset = dataset.map(
    format_batch,
    batched=True,
    remove_columns=dataset["train"].column_names,
 )

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_SEQ_LENGTH)

tokenized_dataset = formatted_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
 )

print(formatted_dataset)

In [ ]:
print(render_conversation(dataset["train"][10]["question"], dataset["train"][10]["answer"]))

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=200,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    lr_scheduler_type="cosine",
    report_to="none",
    bf16=USE_BF16,
    fp16=torch.cuda.is_available() and not USE_BF16,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
    seed=42,
 )

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["val"],
    data_collator=data_collator,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )
    ],
 )

train_result = trainer.train()
train_result.metrics

In [ ]:
eval_metrics = trainer.evaluate()
eval_metrics

In [ ]:
FastLanguageModel.for_inference(model)

question = dataset["val"][0]["question"]
reference_answer = dataset["val"][0]["answer"]
prompt = render_conversation(question, answer=None, add_generation_prompt=True)
model_device = next(model.parameters()).device
inputs = tokenizer(prompt, return_tensors="pt").to(model_device)

with torch.inference_mode():
    generated = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
        repetition_penalty=1.1,
    )

generated_tokens = generated[0][inputs["input_ids"].shape[1]:]
prediction = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("Question:\n")
print(question)
print("\nModel answer:\n")
print(prediction)
print("\nReference answer:\n")
print(reference_answer)

In [ ]:
final_adapter_dir = OUTPUT_DIR / "final_adapter"
final_adapter_dir.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(final_adapter_dir)
tokenizer.save_pretrained(final_adapter_dir)

print(f"Saved LoRA adapter to: {final_adapter_dir.resolve()}")
print("Use this directory when loading the adapter into vLLM as medqa-lora.")